# 1주차 보충 · 낯선 문제를 읽고 예측까지

## 무엇을 할까요?

문제에서 **입력·출력·허용 조건**을 찾고, 숫자 표에서 특성과 정답을 구분합니다. 훈련 자료로 결측 처리와 작은 분류 모델을 실제로 배우고, 별도 검증에서 예측을 점검한 뒤 요구된 순서의 제출 형식을 확인합니다. 모든 표와 숫자는 이 교안을 위해 만든 합성 자료입니다.

본 교안의 표·평균·Python 기초와 [표에서 점수와 ID까지](week01_axes_linear_bridge.ipynb)의 행과 ID 대응 설명을 먼저 확인하세요. 행렬곱·stack·lookup은 이 자료의 필수 선수가 아닙니다. Python 3, NumPy, pandas, scikit-learn과 Jupyter가 필요합니다. 위에서 아래로 Shift+Enter를 누릅니다. **실행 전에 예측과 이유를 적고**, 출력과 다른 첫 단계를 찾아보세요. 뒤의 빈 셀에는 자신의 코드를 작성하고 받은 도움을 기록합니다.


## 1. 원문과 작은 연습을 구분하기

[IOAI 2025 Antique 공식 문제](https://raw.githubusercontent.com/IOAI-official/IOAI-2025/main/Individual-Contest/Antique/Antique.ipynb)는 옛 그림의 진품 여부를 맞히는 과거 국제 대회 문제입니다. 공식 명세에서 훈련 행의 **앞 5개 열은 특성**, 여섯째 열의 라벨은 진품 1, 복제품 −1, 미상 0입니다. 원문은 훈련·검증·테스트 각각 500개의 독립 표본을 설명합니다. 검증과 테스트에는 라벨 열이 없고 대회 중 직접 접근할 수 없으며, 미상 라벨 표본을 포함한 자료도 활용하도록 요구합니다. 정확도를 채점합니다.

원문은 <code>submission.ipynb</code>가 <code>submission.zip</code>을 만들고 그 안에 <code>submissionA.csv</code>와 <code>submissionB.csv</code>를 넣도록 합니다. 각 CSV는 **헤더 없이 한 행에 예측 −1 또는 1 하나**입니다. 각 입력 행에 대응하는 예측 순서를 지켜야 합니다. 파일명, 행 수, 값, 포맷도 결과의 일부입니다.

아래는 **별도로 만든 2특성 연습**입니다. 진품 사진도, 공식 데이터도 아닙니다. 알려진 훈련 라벨만 이용하는 지도학습 입문이므로 공식 문제의 미상 표본 활용 요구를 만족하거나 Antique 전체를 해결했다고 볼 수 없습니다. 원문의 검증·테스트 자료와 아래 연습용 검증·A/B 입력을 혼동하지 마세요. 대회 문제의 모든 조건은 원문을 다시 읽어야 합니다.


### 별도 합성 문제의 계약

한 행은 수집품 하나입니다. 새 수집품에 대한 예측 **시점에** 다음이 주어집니다.

| 항목 | 역할과 허용 범위 |
|---|---|
| row_id | 입력과 출력의 대응을 찾는 이름. 모델 특성은 아님 |
| texture, color | 두 숫자 특성. 예측 시점에 알려지며 한 값은 결측일 수 있음. 비교를 쉽게 하려고 같은 척도의 합성 점수로 둠 |
| label | 훈련에서 −1 또는 1이면 알려진 정답, 0이면 정답 미상. 예측 이후에만 얻는 정답이므로 특성으로 넣지 않음 |
| 나중 검사 결과 | 예측 시점에는 알 수 없음. 답을 그대로 알려 줄 수 있어 입력으로 쓰면 누수 |

훈련과 검증의 행은 서로 다른 수집품이라고 가정합니다. 이 자료의 A/B 입력에는 정답이 없습니다. **계획:** 훈련에서만 대체값과 모델을 배우고, 검증 정답은 선택한 절차를 확인하는 데 사용합니다. 모델·설정 선택 후에만 별도로 보관된 최종 테스트 정답을 열어 평가합니다. 이 노트북에는 그 최종 정답이 없습니다.

질문: 예측할 때 row_id, 두 특성, label, 나중 검사 중 무엇을 모델에 넣을 수 있나요? label 0을 −1로 바꾸면 무엇을 새로 안다고 가정하는 셈인가요?


In [1]:
import io
import csv
import zipfile
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.neighbors import NearestCentroid
from sklearn.pipeline import Pipeline

feature_cols = ["texture", "color"]
training = pd.DataFrame([
    ("T1", 1, 1, -1),
    ("T2", 3, np.nan, -1),
    ("T3", 7, 3, 1),
    ("T4", 9, 5, 1),
    ("T5", 6, 3, 0),
], columns=["row_id", "texture", "color", "label"])
known_train = training[training["label"].isin([-1, 1])].copy()
print("훈련 행과 미상 표시")
display(training)
print("이번 지도학습 fit에 쓰는 ID:", known_train["row_id"].tolist())
print("라벨 미상으로 목표에서 제외한 ID:",
      training.loc[training["label"] == 0, "row_id"].tolist())


훈련 행과 미상 표시


,row_id,texture,color,label
0,T1,1,1.0,-1
1,T2,3,NaN,-1
2,T3,7,3.0,1
3,T4,9,5.0,1
4,T5,6,3.0,0


이번 지도학습 fit에 쓰는 ID: ['T1', 'T2', 'T3', 'T4']
라벨 미상으로 목표에서 제외한 ID: ['T5']


## 2. 배우는 규칙을 손으로 예측하기

<code>fit</code>은 훈련 자료를 읽고 나중에 사용할 값을 저장합니다. 먼저 **알려진 라벨 훈련 행만** 사용한다고 약속합니다. 결측인 T2의 color를 채우기 위한 평균은 (1+3+5)÷3 = **3**입니다. texture 평균은 (1+3+7+9)÷4 = **5**입니다. T5는 정답이 미상이므로 이 단순 예제의 대체값과 모델 학습에서 제외합니다.

이번에는 설명이 쉬운 **가까운 중심 분류**를 씁니다. 결측을 훈련 평균으로 채운 뒤, 각 라벨에 속하는 벡터들의 열별 평균을 중심으로 둡니다. −1의 중심은 [2,2], 1의 중심은 [8,4]입니다. 새 벡터가 어느 중심에 가까운지 비교합니다.

예를 들어 x=[6,3]에 대해 각 거리의 **제곱**은 (6−2)²+(3−2)²=17, (6−8)²+(3−4)²=5입니다. 양수 17과 5의 제곱근을 취해도 더 작은 쪽은 같으므로 이 예측은 1입니다. 동점 처리나 다른 단위·척도라면 규칙을 먼저 확인해야 합니다.

이 모델은 하나의 예시입니다. 문제와 검증 조건에 맞는 다른 분류기를 쓸 수도 있습니다. 중심 거리는 열의 척도에 영향을 받으며, 작은 합성 자료의 결과를 현실 성능으로 일반화할 수 없습니다.


In [2]:
def new_model():
    # 순서: 훈련에서 평균 대체값 학습 → 대체된 벡터로 두 중심 학습
    return Pipeline([("fill", SimpleImputer(strategy="mean")),
                     ("classifier", NearestCentroid())])

assert set(known_train["label"]) == {-1, 1}
assert not known_train[feature_cols].isna().all().any()
model = new_model()
model.fit(known_train[feature_cols], known_train["label"])
fill_values = model.named_steps["fill"].statistics_
centers = model.named_steps["classifier"].centroids_
class_names = model.named_steps["classifier"].classes_
print("훈련에서 배운 대체값:", fill_values)
print("중심의 라벨 순서:", class_names)
print("라벨별 중심:\n", centers)
assert np.allclose(fill_values, [5, 3])
assert np.array_equal(class_names, [-1, 1])
assert np.allclose(centers, [[2, 2], [8, 4]])


훈련에서 배운 대체값: [5. 3.]
중심의 라벨 순서: [-1  1]
라벨별 중심:
 [[2. 2.]
 [8. 4.]]


### 훈련 자료가 바뀌면 무엇을 배울까요?

아래는 **비교용 훈련 사본**만 바꿔 별도 모델을 fit합니다. T4의 texture를 9에서 11로 바꾸면 정답 1의 texture 중심은 (7+11)÷2=9입니다. 비교용 새 입력 [5.25, 3]에서 예측도 달라지는지 확인합니다. 실제 사용할 모델은 처음 만든 <code>model</code>로 유지합니다. 검증 결과를 보고 두 모델 중 고르는 실험이 아닙니다.


In [3]:
changed_train = known_train.copy()
changed_train.loc[changed_train["row_id"] == "T4", "texture"] = 11
changed_model = new_model().fit(changed_train[feature_cols], changed_train["label"])
print("원래 1의 중심:", model.named_steps["classifier"].centroids_[1])
print("훈련값을 바꾼 1의 중심:",
      changed_model.named_steps["classifier"].centroids_[1])
print("원래 모델 대체값 유지:", model.named_steps["fill"].statistics_)
probe = pd.DataFrame([[5.25, 3]], columns=feature_cols)
original_probe = int(model.predict(probe)[0])
changed_probe = int(changed_model.predict(probe)[0])
print("같은 새 입력의 원래 / 훈련 사본 변경 모델 예측:",
      original_probe, "/", changed_probe)
assert np.allclose(changed_model.named_steps["classifier"].centroids_[1], [9, 4])
assert np.allclose(model.named_steps["classifier"].centroids_[1], [8, 4])
assert (original_probe, changed_probe) == (1, -1)


원래 1의 중심: [8. 4.]
훈련값을 바꾼 1의 중심: [9. 4.]
원래 모델 대체값 유지: [5. 3.]
같은 새 입력의 원래 / 훈련 사본 변경 모델 예측: 1 / -1


## 3. 검증으로 확인하고 최종 정답은 보류하기

아래 V는 훈련과 다른 수집품이고, 교육용으로 검증 정답을 보여 줍니다. 예측 전에 모델과 입력 열을 이미 정했습니다. V3의 실제 라벨은 −1이지만 앞에서 손으로 계산한 예측은 1입니다. 네 개 중 몇 개가 맞을까요?


In [4]:
validation = pd.DataFrame([
    ("V1", 2, 3, -1), ("V2", 8, 3, 1),
    ("V3", 6, 3, -1), ("V4", np.nan, 5, 1),
], columns=["row_id", "texture", "color", "label"])
val_predictions = model.predict(validation[feature_cols])
val_view = validation[["row_id", "label"]].copy()
val_view["prediction"] = val_predictions
val_view["correct"] = val_view["label"] == val_view["prediction"]
display(val_view)
accuracy = val_view["correct"].sum() / len(val_view)
print("검증 정확도:", accuracy)
assert val_predictions.tolist() == [-1, 1, 1, 1]
assert accuracy == 3 / 4


,row_id,label,prediction,correct
0,V1,-1,-1,True
1,V2,1,1,True
2,V3,-1,1,False
3,V4,1,1,True


검증 정확도: 0.75


정확도는 <code>맞은 수 ÷ 평가한 전체 수 = 3/4</code>입니다. V3의 오류도 한 행씩 볼 수 있습니다. 이 네 행은 설명용이므로 3/4로 실제 성능을 추정할 수 없습니다. 검증 결과는 모델 선택에 쓸 수 있지만, 같은 검증을 반복해서 보고 고르면 검증에도 맞춰질 수 있습니다. 최종 테스트로 설정을 고르거나 그 정답을 전처리에 섞으면 마지막 평가가 오염됩니다.

V4의 texture는 훈련에서 배운 평균 5로 채웁니다. 검증에는 새 평균을 배우는 <code>fit</code>이 아니라 저장된 규칙을 적용하는 <code>predict</code>만 합니다. 다음 실험은 검증값만 크게 바꾸어도 훈련에서 배운 숫자는 유지되는지 확인합니다.


In [5]:
before_fill = model.named_steps["fill"].statistics_.copy()
before_centers = model.named_steps["classifier"].centroids_.copy()
changed_validation = validation.copy()
changed_validation.loc[changed_validation["row_id"] == "V1", "texture"] = 2000
_ = model.predict(changed_validation[feature_cols])
print("검증을 바꾼 뒤 대체값:", model.named_steps["fill"].statistics_)
print("검증을 바꾼 뒤 중심:\n", model.named_steps["classifier"].centroids_)
assert np.array_equal(model.named_steps["fill"].statistics_, before_fill)
assert np.array_equal(model.named_steps["classifier"].centroids_, before_centers)


검증을 바꾼 뒤 대체값: [5. 3.]
검증을 바꾼 뒤 중심:
 [[2. 2.]
 [8. 4.]]


## 4. 예측을 ID와 묶어서 제출 순서로 옮기기

이 합성 연습의 A/B는 각기 새로운 입력입니다. 편의를 위해 원하는 출력 순서를 별도 ID 목록으로 줍니다. 실제 명세가 **입력 파일의 원래 순서**를 요구하면 읽은 직후 정렬·변형하기 전에 원래 ID 순서를 보관합니다. 공식 Antique의 출력 CSV에는 ID 열을 넣지 않습니다.

아래 함수는 필요한 특성 열이 빠졌거나 ID가 없거나 중복되면 조용히 행을 버리지 않고 오류를 냅니다. 값 한 칸의 결측은 앞에서 배운 대체 규칙으로 처리합니다. <code>feature_cols</code>만 모델에 넣으므로 정답이나 나중 검사 열은 입력에 끼어들지 않습니다.


In [6]:
def is_missing_id(value):
    if not pd.api.types.is_scalar(value):
        raise ValueError("ID는 하나의 값이어야 함")
    return bool(pd.isna(value)) or (isinstance(value, str) and not value.strip())

def predict_pairs(fitted_model, frame):
    required = ["row_id"] + feature_cols
    missing = [name for name in required if name not in frame.columns]
    if missing:
        raise ValueError("필수 열 없음: " + ", ".join(missing))
    if frame["row_id"].map(is_missing_id).any() or frame["row_id"].duplicated().any():
        raise ValueError("row_id 결측 또는 중복")
    values = frame[feature_cols].apply(pd.to_numeric, errors="raise")
    if np.isinf(values.to_numpy(dtype=float)).any():
        raise ValueError("특성에 무한값은 허용하지 않음")
    labels = fitted_model.predict(values)
    return list(zip(frame["row_id"].tolist(), labels.tolist()))

A = pd.DataFrame([("A08", 8, 4), ("A02", 1, 2), ("A05", 4, np.nan)],
                 columns=["row_id", "texture", "color"])
B = pd.DataFrame([("B09", 9, 4), ("B04", 1, 1)],
                 columns=["row_id", "texture", "color"])
order_A = ["A05", "A08", "A02"]
order_B = ["B04", "B09"]
pairs_A = predict_pairs(model, A)
pairs_B = predict_pairs(model, B)
print("입력 A 행과 예측:", pairs_A)
print("출력해야 하는 A ID 순서:", order_A)


입력 A 행과 예측: [('A08', 1), ('A02', -1), ('A05', -1)]
출력해야 하는 A ID 순서: ['A05', 'A08', 'A02']


### 길이가 같아도 행이 뒤바뀔 수 있습니다

<code>pairs_A</code>는 예측을 만든 행의 ID를 함께 보유합니다. 요구된 순서의 각 ID로 다시 찾아야 합니다. 아래에서 순서만 바꾼 값과 ID를 기준으로 정렬한 값을 비교하세요. 또한 중복·누락·불법 예측값을 확인합니다. 값에 0은 허용하지 않는 **이 연습의 출력 계약**입니다.


In [7]:
def align_predictions(expected_ids, pairs):
    ids = list(expected_ids)
    pairs = list(pairs)
    if any(is_missing_id(row_id) for row_id in ids):
        raise ValueError("요구 ID 결측")
    if len(ids) != len(set(ids)):
        raise ValueError("요구 ID 중복")
    mapping = {}
    for row_id, prediction in pairs:
        if is_missing_id(row_id):
            raise ValueError("예측 ID 결측")
        if row_id in mapping:
            raise ValueError("예측 ID 중복")
        if isinstance(prediction, (bool, np.bool_)) or not isinstance(prediction, (int, np.integer)):
            raise ValueError("예측은 정수 -1 또는 1")
        if int(prediction) not in (-1, 1):
            raise ValueError("예측은 -1 또는 1")
        mapping[row_id] = int(prediction)
    if set(mapping) != set(ids):
        raise ValueError("요구 ID와 예측 ID가 다름")
    return [label for _, label in pairs]

ordered_A = align_predictions(order_A, pairs_A)
ordered_B = align_predictions(order_B, pairs_B)
naive_A = [label for _, label in pairs_A]
print("A 입력 순서만 그대로 쓴 잘못된 값:", naive_A)
print("A 요구 ID 순서로 재배열한 값:", ordered_A)
print("B 요구 ID 순서로 재배열한 값:", ordered_B)
assert ordered_A == [-1, 1, -1]
assert ordered_B == [-1, 1]
assert naive_A != ordered_A


A 입력 순서만 그대로 쓴 잘못된 값: [1, -1, -1]
A 요구 ID 순서로 재배열한 값: [-1, 1, -1]
B 요구 ID 순서로 재배열한 값: [-1, 1]


### ZIP 안의 실제 CSV를 열어 검사하기

아래는 메모리 안에 실제 ZIP과 CSV 바이트를 만들고 다시 엽니다. 연습용이므로 파일을 서버에 제출하지 않습니다. 두 CSV의 파일명, 각 행의 열 수·라벨·개수와, 앞에서 ID에 묶은 예측대로 배치됐는지를 검사합니다. 헤더나 ID 열을 넣으면 이 형식에서는 오류입니다.

ID가 없는 CSV 자체만 보면 어느 수집품의 예측인지 알 수 없습니다. 아래 순서 검사는 따로 보관한 **ID→예측 대응표**에 대조하는 것입니다. 대응표 자체가 잘못 만들어졌다면 CSV 형식 검사만으로는 발견할 수 없으므로, 예측할 때 행과 ID를 함께 옮기는지와 순서 변경 전후 결과도 확인해야 합니다.


In [8]:
def csv_text(labels):
    stream = io.StringIO(newline="")
    writer = csv.writer(stream, lineterminator="\n")
    writer.writerows([[value] for value in labels])
    return stream.getvalue()

def archive_texts(text_A, text_B):
    stream = io.BytesIO()
    with zipfile.ZipFile(stream, mode="w", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.writestr("submissionA.csv", text_A)
        archive.writestr("submissionB.csv", text_B)
    return stream.getvalue()

def make_archive(ids_A, preds_A, ids_B, preds_B):
    a = align_predictions(ids_A, preds_A)
    b = align_predictions(ids_B, preds_B)
    return archive_texts(csv_text(a), csv_text(b))

def inspect_archive(blob, ids_A, preds_A, ids_B, preds_B):
    wanted = {"submissionA.csv": align_predictions(ids_A, preds_A),
              "submissionB.csv": align_predictions(ids_B, preds_B)}
    with zipfile.ZipFile(io.BytesIO(blob)) as archive:
        names = archive.namelist()
        if len(names) != 2 or set(names) != set(wanted):
            raise ValueError("ZIP 파일명이 다르거나 중복됨")
        actual = {}
        for name, expected in wanted.items():
            rows = list(csv.reader(io.StringIO(archive.read(name).decode("utf-8"))))
            if any(len(row) != 1 or row[0] not in ("-1", "1") for row in rows):
                raise ValueError("헤더 없는 한 열의 -1 또는 1이 필요함")
            labels = [int(row[0]) for row in rows]
            if len(labels) != len(expected):
                raise ValueError("입력 행 수와 예측 행 수가 다름")
            if labels != expected:
                raise ValueError("저장한 ID 대응 순서가 다름")
            actual[name] = labels
    return actual

archive_bytes = make_archive(order_A, pairs_A, order_B, pairs_B)
print("ZIP 검사 결과:", inspect_archive(archive_bytes, order_A, pairs_A, order_B, pairs_B))
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    print("A CSV 원문:", repr(archive.read("submissionA.csv").decode("utf-8")))


ZIP 검사 결과: {'submissionA.csv': [-1, 1, -1], 'submissionB.csv': [-1, 1]}
A CSV 원문: '-1\n1\n-1\n'


### 실제 연습 파일을 쓰고 다시 열기

이번에는 같은 합성 바이트를 임시 작업 폴더의 <code>submission.zip</code>에 씁니다. 디스크의 파일을 다시 읽어 ZIP 구조와 CSV 값을 검사하고, 작업이 끝나면 임시 폴더와 파일을 자동으로 지웁니다. 이 노트북은 연습용입니다. 실제 공식 제출에서는 원문이 요구한 <code>submission.ipynb</code>와 지정된 작업 디렉터리·<code>submission.zip</code> 파일명을 따르며, 공식 데이터로 정확한 전체 계약을 따로 확인해야 합니다.


In [9]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory(prefix="toy-submission-") as directory:
    practice_zip = Path(directory) / "submission.zip"
    practice_zip.write_bytes(archive_bytes)
    read_back = practice_zip.read_bytes()
    verified = inspect_archive(read_back, order_A, pairs_A, order_B, pairs_B)
    print("임시 작업 폴더에 저장한 파일:", practice_zip.name)
    print("파일을 다시 열어 확인한 두 CSV:", verified)
    assert read_back == archive_bytes
# 블록이 끝나면 임시 파일은 삭제됩니다.
print("임시 파일 정리:", not practice_zip.exists())
assert not practice_zip.exists()


임시 작업 폴더에 저장한 파일: submission.zip
파일을 다시 열어 확인한 두 CSV: {'submissionA.csv': [-1, 1, -1], 'submissionB.csv': [-1, 1]}
임시 파일 정리: True


### 실수한 입력을 직접 확인하기

어떤 조건이 깨지는지 먼저 예상하세요. 다음 셀은 오류를 잡아 이름을 보여 주므로 이후에도 실행할 수 있습니다. 값이 결측인 행과 열 자체가 없는 경우, 순서가 다른 값과 개수가 다른 경우를 구별하세요.


In [10]:
def show_rejected(name, action):
    try:
        action()
    except ValueError as error:
        print(name, "→", type(error).__name__, ":", error)
    else:
        raise AssertionError(name + " 오류를 놓쳤습니다")

show_rejected("필수 열 누락", lambda: predict_pairs(model, A.drop(columns=["color"])))
show_rejected("요구 ID 결측", lambda: align_predictions([None], [(None, 1)]))
show_rejected("예측 ID 결측", lambda: align_predictions(["A"], [(np.nan, 1)]))
show_rejected("예측 ID 중복", lambda: align_predictions(order_A, pairs_A + [pairs_A[0]]))
show_rejected("미상 0을 출력", lambda: align_predictions(order_A, [("A05", 0), ("A08", 1), ("A02", -1)]))
wrong_order_zip = archive_texts(csv_text(naive_A), csv_text(ordered_B))
show_rejected("같은 개수의 순서 오류",
              lambda: inspect_archive(wrong_order_zip, order_A, pairs_A, order_B, pairs_B))
header_zip = archive_texts("label\n" + csv_text(ordered_A), csv_text(ordered_B))
show_rejected("헤더 추가",
              lambda: inspect_archive(header_zip, order_A, pairs_A, order_B, pairs_B))
short_zip = archive_texts(csv_text(ordered_A[:-1]), csv_text(ordered_B))
show_rejected("예측 행 누락",
              lambda: inspect_archive(short_zip, order_A, pairs_A, order_B, pairs_B))


필수 열 누락 → ValueError : 필수 열 없음: color
요구 ID 결측 → ValueError : 요구 ID 결측
예측 ID 결측 → ValueError : 예측 ID 결측
예측 ID 중복 → ValueError : 예측 ID 중복
미상 0을 출력 → ValueError : 예측은 -1 또는 1
같은 개수의 순서 오류 → ValueError : 저장한 ID 대응 순서가 다름
헤더 추가 → ValueError : 헤더 없는 한 열의 -1 또는 1이 필요함
예측 행 누락 → ValueError : 입력 행 수와 예측 행 수가 다름


## 5. 다른 명세의 문제를 혼자 풀기

**위 완성 예제와 별도 교사용 해설을 가리고** 입력·출력·정답 규칙부터 다시 적으세요.

### 새 합성 명세 · 세 특성의 기기 점검

행은 기기 하나입니다. 예측 시점에 주어지는 수치 특성은 <code>vibration, sound, heat</code> 세 개이며 단위 없는 같은 척도 점수입니다. 이 문제의 훈련 라벨은 **0=정상, 1=점검, −1=미상**입니다. 앞 문제와 0, −1의 뜻이 바뀌었습니다. 미상 행을 정상이나 점검으로 바꾸지 않습니다. 검증의 정답은 모델을 정한 다음 확인합니다.

이 문제의 출력은 ZIP이 아닌 <code>audit.csv</code> 한 개입니다. 첫 줄은 정확히 <code>item,signal</code>이고 그 뒤 각 행은 **제시한 입력 순서**의 item ID와 0 또는 1 예측입니다. 세 특성 중 값 한 칸이 비면 라벨이 알려진 훈련 행에서 배운 각 열의 평균으로 채웁니다.

1. 훈련에서 쓸 ID, 제외할 미상 ID, 모델 입력 열과 사용할 수 없는 정답을 나누세요. 정답과 나중 검사 결과가 예측용 입력이 아닌 이유도 설명하세요.
2. 알려진 훈련 행으로 각 특성의 대체 평균을 손계산하세요. 타당한 학습 분류기를 하나 선택해 이 데이터에 새로 fit하고, 모델이 무엇을 배웠는지 값으로 확인·설명하세요. 예를 들어 가까운 중심 분류기를 선택했다면 라벨별 중심을 손계산해 비교할 수 있습니다. 다른 분류기를 선택했다면 그 모델에 맞는 학습 결과를 확인하세요.
3. 별도 검증 세 행의 예측과 정확도를 계산하세요. 최종 테스트 정답은 없고 선택 후까지 보류한다는 뜻을 설명하세요.
4. 새로운 입력의 ID를 그대로 유지해 헤더가 있는 CSV를 작성하고 다시 읽어 행 수·열 이름·ID 순서·허용값을 검사하세요. 입력 열 누락과 ID 중복을 각각 발견하면 어떻게 할지도 코드와 말로 보여 주세요. 완성 예제를 그대로 복사하지 말고 이 명세에 맞게 바꿉니다.
5. <code>label=-1</code>을 진짜 정답 −1로 학습하거나, 예측만 거꾸로 정렬하고 ID를 그대로 두면 무엇이 잘못되나요? 값 한 칸의 결측과 열 전체 누락도 구분하세요.


In [11]:
exercise_train = pd.DataFrame([
    ("R1", 0, 2, 1, 0), ("R2", 2, np.nan, 1, 0),
    ("R3", 6, 4, 3, 1), ("R4", 8, 6, 3, 1),
    ("R5", 4, 5, 2, -1),
], columns=["item", "vibration", "sound", "heat", "label"])
exercise_validation = pd.DataFrame([
    ("V1", 2, 3, 1, 0), ("V2", 6, 5, 3, 1),
    ("V3", 5, 4, 1, 0),
], columns=["item", "vibration", "sound", "heat", "label"])
exercise_input = pd.DataFrame([
    ("J3", 7, 5, 3), ("J1", 1, 3, 1), ("J2", 2, np.nan, 1),
], columns=["item", "vibration", "sound", "heat"])


내가 읽은 입력·출력·미상 규칙:

손계산한 훈련 평균 / 선택한 모델의 배운 값과 검증 예측·정확도:

작성한 CSV의 확인과 오류 처리:

달라진 생각 / 받은 도움(없음·힌트·예시·직접 설명):


In [12]:
# 새 명세에 맞는 코드와 검증을 직접 작성하세요.


## 다음에 연결하기

미상 데이터를 활용하는 모델은 나중에 배웁니다. 같은 숫자라도 도구의 약속이 다를 수 있습니다. 예를 들어 scikit-learn의 LabelSpreading/LabelPropagation에서 −1은 **미상**으로 쓰입니다. Antique의 −1은 **복제품**, 0은 미상이므로 원문 라벨을 그대로 넣으면 뜻이 뒤바뀝니다. 사용할 때는 예를 들어 복제품 −1→0, 진품 1→1, 미상 0→−1로 바꾸고, 결과의 0→−1, 1→1로 되돌리는 대응을 명시해야 합니다. 이 예시는 라벨 대응만 설명하며 반지도 학습이나 공식 문제를 풀지 않습니다.

[2주차 예측과 평가](../week02/week02_prediction_evaluation.ipynb)에서는 예측 시점, 분할, 학습형 전처리와 평가지표를 더 다룹니다. 10주차에는 처음 보는 자료에서 그 문제의 입력·출력과 검증 조건을 다시 읽고 코드를 작성합니다.

## 참고 원문과 범위

- [KOAI 2027 안내](https://kitpa.org/koai/2027)는 국내 대회 안내이고, 이 안내가 연결한 [KOAI/IOAI 실라버스](https://kitpa.org/koai/syllabus)는 확인 당시 IOAI 2026 기준으로 표시됐습니다.
- [IOAI 2025 Antique 문제와 baseline](https://raw.githubusercontent.com/IOAI-official/IOAI-2025/main/Individual-Contest/Antique/Antique.ipynb)은 과거 국제 개인전 원문입니다. KOAI 2027 중등부의 실제 문제나 위 합성 문제의 정답이 아닙니다. 공식 문제는 5특성과 미상 표본 활용을 요구합니다.
- [scikit-learn 가까운 중심 분류기](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.NearestCentroid.html), [데이터 누수 방지](https://scikit-learn.org/stable/common_pitfalls.html), [반지도 라벨 표기](https://scikit-learn.org/stable/modules/semi_supervised.html)에서 도구의 조건을 확인하세요. 공식 페이지와 GitHub main은 달라질 수 있으므로 사용 시 다시 확인합니다.
